# 사후(post-hoc) 실험: 문헌 방식 CNN 재현 (N-CMAPSS DS02)

**위치.** Phase 3(사전 등록 평가) 이후에 추가한 실험입니다. 원고에는 "post-hoc, not pre-registered"로 표시합니다.
**이 노트북을 실행하기 전에 GitHub 저장소에 커밋하세요.** 그래야 계획이 결과보다 앞섰다는 기록이 남습니다.

**목적.** 문헌의 DS02 결과(Arias Chao et al., RESS 2022: 순수 CNN 4.95, 표본 단위)와 우리 비교모델(1 Hz 원시 CNN 8.78, 사이클 단위) 사이의 차이를 설명합니다. 두 가지를 확인합니다.
1. 문헌에 가까운 학습 조건이면 데이터 기반 CNN이 DS02에서 PI-BDP보다 정확한가?
2. 표본 단위 채점과 사이클 단위 채점의 차이가 결과를 얼마나 바꾸는가?

**설정 (실행 전 고정, test 결과를 보고 바꾸지 않음)**
- **데이터:** Phase 0의 0.1 Hz 데이터. DS02 dev 6대(2, 5, 10, 16, 18, 20)로 학습하고, test는 11, 14, 15입니다.
- **입력:** 운전조건 W 4개와 측정센서 X_s 14개, 총 18채널. 원문의 순수 데이터 기반 설정 x = [w, x_s]와 같습니다. 가상센서와 건전성 매개변수는 쓰지 않습니다.
- **윈도:** 길이 50 표본(0.1 Hz, 약 500초). 엔진 궤적 안에서 1표본 간격으로 자르고, 레이블은 윈도 마지막 표본의 RUL(사이클, 상한 없음)입니다.
- **원문 설정을 그대로 따릅니다** (Arias Chao et al., RESS 217 (2022) 107961, 4.3~4.5절).
  - 정규화: dev 전체의 최소·최대로 [−1, 1] min-max
  - 모델: 합성곱 3층(커널 10, zero padding, ReLU). 채널은 앞 두 층이 10, 마지막 층이 1입니다. 그다음 flatten → 완전연결 50(ReLU) → 선형 출력 1
  - 초기화: Xavier
  - 최적화: AMSgrad, 학습률 0.001, 배치 1024
  - 학습: 최대 30 epoch, 조기종료 patience 5. 검증 집합은 dev 윈도의 무작위 10%입니다.
- **시드:** 5개
- **채점:** 같은 예측을 두 방식으로 채점합니다.
  - (a) 표본 단위: test 윈도 전체의 RMSE (문헌 방식)
  - (b) 사이클 단위: 사이클 안 윈도 예측의 평균으로 RMSE (우리 프로토콜)
- **PI-BDP V0 환산:** Phase 3에 저장된 V0 예측도 두 방식으로 다시 집계합니다. 표본 단위는 사이클마다 표본 수로 가중합니다.

**선택 사항.** `RUN_ALL = True`로 바꾸면 9개 서브셋 전부에 같은 절차를 적용합니다(시드 3개, 수 시간 소요). 먼저 DS02만 돌린 뒤 결정하세요.

**런타임:** T4 GPU. DS02 시드 5개에 약 1~1.5시간 걸릴 것으로 예상합니다. 끝나면 `repro_summary.txt`를 Claude에게 올려 주세요.

In [ ]:
import os, sys, json, glob, time, hashlib
import numpy as np, pandas as pd, torch, torch.nn as nn
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/PI-BDP_NCMAPSS'
else:
    ROOT = os.environ.get('NCM_ROOT', './ncm')
OUT = f'{ROOT}/posthoc_cnn'; os.makedirs(OUT, exist_ok=True)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; print('장치:', DEV)
L, BATCH, LR, MAX_EP, PATIENCE, N_SEEDS, VAL_FRAC = 50, 1024, 1e-3, 30, 5, 5, 0.10   # 원문 4.3~4.5절
RUN_ALL, N_SEEDS_ALL = False, 3

In [ ]:
# === 데이터: 0.1 Hz dev/test (Phase 0 저장본) ===
def load(s, part):
    f = f'{ROOT}/phase0/{"dev" if part == "dev" else "sealed_test"}_{s}.npz'
    d = np.load(f, allow_pickle=False); an = list(d['A_var'])
    X = np.hstack([d['W'], d['X_s']]).astype(np.float32)
    u = d['A'][:, an.index('unit')].astype(int); c = d['A'][:, an.index('cycle')].astype(int)
    d.close()
    eol = pd.Series(c).groupby(u).max()
    return X, u, c, eol

def window_ends(u, units):
    # 각 엔진 안에서 L-1번째 표본부터 끝까지 (윈도가 엔진 경계를 넘지 않음)
    out = []
    for k in units:
        idx = np.flatnonzero(u == k)
        if len(idx) >= L: out.append(idx[L - 1:])
    return np.concatenate(out)

class CNN(nn.Module):
    def __init__(self, cin=18):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv1d(cin, 10, 10, padding='same'), nn.ReLU(),
                                  nn.Conv1d(10, 10, 10, padding='same'), nn.ReLU(),
                                  nn.Conv1d(10, 1, 10, padding='same'), nn.ReLU())     # 마지막 합성곱은 1채널 (원문)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(1 * L, 50), nn.ReLU(), nn.Linear(50, 1))
        for m in self.modules():                                                        # Xavier 초기화 (원문)
            if isinstance(m, (nn.Conv1d, nn.Linear)): nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x):                    # x: (B, L, C)
        return self.head(self.conv(x.transpose(1, 2))).squeeze(-1)

AR = torch.arange(L, device=DEV)
def predict(net, Xg, ends):
    net.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(ends), 8192):
            e = ends[i:i + 8192]; out.append(net(Xg[e[:, None] - L + 1 + AR]).cpu().numpy())
    return np.concatenate(out)

def train(Xg, Yg, tr_ends, va_ends, y_va, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    net = CNN(Xg.shape[1]).to(DEV); opt = torch.optim.Adam(net.parameters(), lr=LR, amsgrad=True)
    g = torch.Generator().manual_seed(seed); best, best_state, bad, ep_used = np.inf, None, 0, 0
    for ep in range(MAX_EP):
        net.train(); perm = tr_ends[torch.randperm(len(tr_ends), generator=g).to(DEV)]
        for i in range(0, len(perm), BATCH):
            e = perm[i:i + BATCH]; opt.zero_grad()
            ((net(Xg[e[:, None] - L + 1 + AR]) - Yg[e]) ** 2).mean().backward(); opt.step()
        v = float(np.sqrt(np.mean((predict(net, Xg, va_ends) - y_va) ** 2)))
        if v < best - 1e-4: best, best_state, bad, ep_used = v, {k: x.clone() for k, x in net.state_dict().items()}, 0, ep + 1
        else: bad += 1
        if bad >= PATIENCE: break
    net.load_state_dict(best_state); return net, best, ep_used
print('정의 완료')

In [ ]:
# === 실행: 서브셋별 (DS02 기본, RUN_ALL이면 전체) ===
cfg = json.load(open(f'{ROOT}/phase1/config.json'))
SUBS = sorted(cfg['k']) if RUN_ALL else ['DS02-006']
for s in SUBS:
    nseed = N_SEEDS if s == 'DS02-006' else N_SEEDS_ALL
    Xd, ud, cd, eold = load(s, 'dev'); Xt, ut, ct, eolt = load(s, 'test')
    cens = eold >= 100
    dev_units = [k for k in sorted(eold.index) if not cens[k]]
    devm = np.isin(ud, dev_units); lo, hi = Xd[devm].min(0), Xd[devm].max(0)   # dev 전체 min-max (원문)
    Xall = np.vstack([Xd, Xt]); uall = np.r_[ud, ut + 10000]; call = np.r_[cd, ct]   # test 엔진 번호를 겹치지 않게 이동
    yall = np.r_[eold.reindex(ud).values - cd, eolt.reindex(ut).values - ct].astype(np.float32)
    Xg = torch.tensor(2 * (Xall - lo) / (hi - lo + 1e-6) - 1, device=DEV); Yg = torch.tensor(yall, device=DEV)   # [-1, 1]
    all_e = window_ends(uall, dev_units); rs = np.random.default_rng(0).permutation(len(all_e))
    nva = int(round(VAL_FRAC * len(all_e))); va_np = np.sort(all_e[rs[:nva]]); tr_np = np.sort(all_e[rs[nva:]])
    tr_e = torch.tensor(tr_np, device=DEV); va_e = torch.tensor(va_np, device=DEV)   # 검증: dev 윈도 무작위 10% (원문)
    te_np = window_ends(uall, [k + 10000 for k in sorted(eolt.index)]); te_e = torch.tensor(te_np, device=DEV)
    print(f'{s}: 학습 윈도 {len(tr_e):,}, 검증 윈도 {len(va_np):,}, test 윈도 {len(te_np):,}')
    for sd in range(nseed):
        f = f'{OUT}/pred_{s}_seed{sd}.csv.gz'
        if os.path.exists(f): print('있음:', s, sd); continue
        t0 = time.time()
        net, vbest, ep = train(Xg, Yg, tr_e, va_e, yall[va_np], sd)
        p = predict(net, Xg, te_e)
        pd.DataFrame({'subset': s, 'unit': uall[te_np] - 10000, 'cycle': call[te_np], 'pred': p, 'true': yall[te_np], 'seed': sd,
                      'val_rmse': vbest, 'epochs': ep}).to_csv(f, index=False)
        print(f'  seed {sd}: val {vbest:.2f}, epochs {ep}, 표본 RMSE {np.sqrt(np.mean((p - yall[te_np]) ** 2)):.3f} ({time.time()-t0:.0f}초)')
    del Xg, Yg; torch.cuda.empty_cache()

In [ ]:
# === 채점: 표본 단위 대 사이클 단위, PI-BDP V0 환산 포함 ===
rows, unit_rows = [], []
done = sorted({os.path.basename(p).split('_seed')[0][5:] for p in glob.glob(f'{OUT}/pred_*_seed*.csv.gz')})
for s in done:
    R = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(f'{OUT}/pred_{s}_seed*.csv.gz'))])
    samp, cyc = [], []
    for sd, g in R.groupby('seed'):
        samp.append(np.sqrt(np.mean((g.pred - g.true) ** 2)))
        gc = g.groupby(['unit', 'cycle'], as_index=False)[['pred', 'true']].mean()
        cyc.append(np.sqrt(np.mean((gc.pred - gc.true) ** 2)))
    # PI-BDP V0: 사이클 예측 → 표본 수 가중 (같은 test 윈도 끝 표본 기준)
    v0 = pd.read_csv(f'{ROOT}/phase3/test_PI-BDP_V0_{s}.csv.gz')[['unit', 'cycle', 'q500', 'true']]
    w = R[R.seed == R.seed.min()].groupby(['unit', 'cycle']).size().rename('n').reset_index()
    vm = v0.merge(w, on=['unit', 'cycle'])
    v0_samp = float(np.sqrt(np.average((vm.q500 - vm.true) ** 2, weights=vm.n)))
    v0_cyc = float(np.sqrt(np.mean((v0.q500 - v0.true) ** 2)))
    rows.append(dict(subset=s, seeds=R.seed.nunique(),
                     cnn_sample_rmse_mean=float(np.mean(samp)), cnn_sample_rmse_sd=float(np.std(samp)),
                     cnn_cycle_rmse_mean=float(np.mean(cyc)), cnn_cycle_rmse_sd=float(np.std(cyc)),
                     v0_sample_rmse=v0_samp, v0_cycle_rmse=v0_cyc,
                     epochs_mean=float(R.groupby('seed').epochs.first().mean())))
    for k, g in R.groupby('unit'):
        e = [np.sqrt(np.mean((gg.pred - gg.true) ** 2)) for _, gg in g.groupby('seed')]
        vu_ = vm[vm.unit == k]
        unit_rows.append(dict(subset=s, unit=k, cnn_sample_rmse=float(np.mean(e)),
                              v0_sample_rmse=float(np.sqrt(np.average((vu_.q500 - vu_.true) ** 2, weights=vu_.n)))))
T = pd.DataFrame(rows); U = pd.DataFrame(unit_rows)
T.to_csv(f'{OUT}/repro_table.csv', index=False); U.to_csv(f'{OUT}/repro_units.csv', index=False)
txt = '\n'.join([f"Post-hoc literature-style CNN reproduction {time.strftime('%Y-%m-%d %H:%M:%S')} (NOT pre-registered)",
                 'Reference (Arias Chao et al., RESS 2022, DS02, sample-level): CNN 4.95 ± 0.15 (as reported; to be verified against the PDF)', '',
                 T.round(3).to_string(index=False), '', 'Per test unit (sample-level, CNN mean over seeds):', U.round(3).to_string(index=False)])
open(f'{OUT}/repro_summary.txt', 'w', encoding='utf-8').write(txt)
H = {os.path.basename(p): hashlib.sha256(open(p, 'rb').read()).hexdigest() for p in sorted(glob.glob(f'{OUT}/*')) if 'sha256' not in p}
json.dump(H, open(f'{OUT}/sha256_posthoc.json', 'w'), indent=1)
print(txt)